# FCA–MoE F2-E — Optimized, chunked confirmatory campaign

This notebook runs the optimized F2-E implementation. It reuses the previously completed temporal LOO validation only after checking the LOO script hash, data hashes, and every validation setting. The nested bootstrap is a new campaign with a new code fingerprint and a separate output directory.

Optimizations preserve the 999 outer × 100 inner design, eight screening replicates, 0.1 temporal grid, three lambda candidates, and all six FCA metrics. The optimized implementation uses one FCA traversal per context across support thresholds, cached invariant transition structures, deterministic partial-screen resume, inner-null checkpoints, and chunked outer summaries.

Do not interpret partial chunks as final inferential results. A final result is available only after all 999 outer replicates succeed and the audit artifacts have been checked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess
REPO = Path('/content/FCA-MoE')
BRANCH = 'research/fca-moe-v0.3-f2e-optimized-chunked'
REPO_URL = 'https://github.com/msellamiTN/FCA-MoE.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
# Always fetch the requested branch and reset this runtime checkout to that exact FETCH_HEAD.
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
actual_branch = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
actual_sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Checked-out branch:', actual_branch)
print('Checked-out commit:', actual_sha)
if actual_branch != BRANCH:
    raise RuntimeError(f'Expected branch {BRANCH}, got {actual_branch}')
%cd /content/FCA-MoE/research/v03
!python -m pip install -e .
!python -m pip install pytest


In [ ]:
# Full v0.3 test suite, including transition-kernel normalization, invariance, detailed balance and exact transition likelihood.
!python -m pytest -q


In [ ]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/FCA_MoE_v03/qwen15_moe_a27_wikitext_test/contexts_v1')
PREVIOUS_CAMPAIGN = ROOT/'sequence_replication/layer_00/temporal/f2e_confirmatory_campaign_v1'
LOO_SUMMARY = PREVIOUS_CAMPAIGN/'temporal_loo/loo_validation_summary.json'
CAMPAIGN_OUT = ROOT/'sequence_replication/layer_00/temporal/f2e_confirmatory_campaign_v2_optimized'
required = [ROOT/'qwen_contexts_by_layer.npz', ROOT/'sequence_alignment.csv', LOO_SUMMARY]
for path in required:
    print(path, path.exists())
    if not path.exists(): raise FileNotFoundError(path)
print('Optimized campaign output:', CAMPAIGN_OUT)
print('Reusing validated LOO prerequisite:', LOO_SUMMARY)


## Optimized confirmatory execution

The runner verifies that the previous LOO summary matches the unchanged LOO validator, source context, alignment, lambda grid, temporal grid, screening count, predictive-replicate count, and seed. It then starts a new optimized nested bootstrap under a fresh fingerprint.

Run once with `RESUME = False`. After a runtime interruption, set `RESUME = True` and rerun this cell with the same paths and arguments. Do not change statistical settings when resuming.

In [ ]:
import subprocess, sys, json

# First execution: False. After an actual nested-bootstrap interruption: True.
RESUME = False

# Guard against running an old checkout or a stale CLI.
actual_branch = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
actual_sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Execution branch:', actual_branch)
print('Execution commit:', actual_sha)
if actual_branch != BRANCH:
    raise RuntimeError(f'Wrong checkout: expected {BRANCH}, got {actual_branch}. Rerun the checkout/installation cell.')

runner = 'scripts/run_f2e_confirmatory_campaign.py'
help_result = subprocess.run([sys.executable, runner, '--help'], text=True, capture_output=True)
if help_result.returncode != 0:
    print(help_result.stdout)
    print(help_result.stderr)
    raise RuntimeError('Campaign runner --help failed; check the checkout/import environment.')
required_flags = ['--screen-checkpoint-every', '--inner-checkpoint-every', '--outer-chunk-size', '--checkpoint-every']
missing_flags = [flag for flag in required_flags if flag not in help_result.stdout]
if missing_flags:
    raise RuntimeError(f'Checked-out campaign runner does not support {missing_flags}; actual commit is {actual_sha}')

cmd = [
    sys.executable, runner,
    '--root', str(ROOT),
    '--output-base', str(CAMPAIGN_OUT),
    '--reuse-temporal-loo-summary', str(LOO_SUMMARY),
    '--outer-replicates', '999',
    '--inner-replicates', '100',
    '--screen-replicates', '8',
    '--weight-grid-step', '0.1',
    '--lambdas', '0.0003', '0.001', '0.003',
    '--maxiter', '400',
    '--predictive-replicates', '200',
    '--screen-checkpoint-every', '25',
    '--inner-checkpoint-every', '10',
    '--outer-chunk-size', '25',
    '--checkpoint-every', '1',
    '--seed', '20261120',
]
if RESUME:
    cmd.append('--resume')
print('Launching:', ' '.join(cmd), flush=True)

# Stream all child output live and preserve it in Drive for diagnosis.
CAMPAIGN_OUT.mkdir(parents=True, exist_ok=True)
console_log = CAMPAIGN_OUT/'campaign_console.log'
with console_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()

print('Campaign process exit code:', return_code)
print('Full console log:', console_log)
manifest_path = CAMPAIGN_OUT/'campaign_manifest.json'
if return_code != 0:
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        loo = manifest.get('temporal_loo_result') or {}
        print('Campaign manifest diagnosis:', json.dumps({
            'stage': manifest.get('stage'),
            'status': manifest.get('status'),
            'temporal_loo_reused_from_cache': manifest.get('temporal_loo_reused_from_cache'),
            'loo_status': loo.get('status'),
            'predeclared_loo_diagnostic_pass': loo.get('predeclared_loo_diagnostic_pass'),
            'mean_transition_nll_gain_selected_vs_iid': loo.get('mean_transition_nll_gain_selected_vs_iid'),
            'folds_better_than_iid': loo.get('folds_better_than_iid'),
            'nested_bootstrap_output': manifest.get('nested_bootstrap_output'),
        }, indent=2))
    else:
        print('No campaign manifest was written. The failure occurred before the runner reached its manifest stage.')
    raise RuntimeError(f'Confirmatory campaign stopped with exit code {return_code}. Inspect console log and manifest above.')
print('Campaign execution completed successfully.')


In [ ]:
## Live and resumable output inspection

Use this cell when the campaign execution cell has stopped or after a checkpoint has been written. It reports the campaign stage, outer replicate count, partial temporal screens, inner-null progress, and final inferential summary.

from pathlib import Path
import json, pandas as pd

NESTED = CAMPAIGN_OUT/'nested_bootstrap'
manifest_path = CAMPAIGN_OUT/'campaign_manifest.json'
progress_path = NESTED/'progress.json'
outer_csv = NESTED/'outer_bootstrap_replicates.csv'

for label, path in [('Campaign manifest', manifest_path), ('Nested progress', progress_path), ('Outer checkpoints', outer_csv), ('Final summary', NESTED/'f2e_nested_bootstrap_summary.json')]:
    print(f'\\n=== {label} ===')
    if path.exists():
        if path.suffix == '.json': print(json.dumps(json.loads(path.read_text()), indent=2))
        elif path.suffix == '.csv':
            frame = pd.read_csv(path)
            print(frame.tail(10).to_string(index=False))
            print('Rows:', len(frame))
    else: print('Not created yet:', path)

screen_partials = sorted((NESTED/'outer_weight_screens').glob('*.partial.csv')) if (NESTED/'outer_weight_screens').exists() else []
print('\\nPartial temporal screens:', len(screen_partials))
for path in screen_partials[-5:]:
    print(path.name, 'candidates:', len(pd.read_csv(path)))

inner_progress = sorted((NESTED/'outer_inner_metrics').glob('*.progress.json')) if (NESTED/'outer_inner_metrics').exists() else []
print('Inner-null progress files:', len(inner_progress))
for path in inner_progress[-5:]:
    print(path.name, json.loads(path.read_text()))


## Output package and interpretation guardrails

The output directory contains model fits, candidate weight screens, inner-null metrics and diagnostics, per-replicate parameters, outer chunks, exact checkpoint state, campaign hashes, and the final nested-bootstrap summary.

The four-fold LOO diagnostic is predictive and descriptive, not a hypothesis test. The nested inference remains conditional on the fitted pairwise-temporal null family. Any failed outer replicate prevents confirmatory reporting until resolved. Even a significant result supports excess higher-order FCA structure only for the evaluated model, dataset, sample, and layer; it does not prove semantic specialization or causal interpretability.